# A Deep Agent is just a regular agent with
- A small collection of essential tools
- A small collection of prebuilt middleware

### a few superpowers
- The ability to read from and write to filesystems
- The ability to spawn (sync/async) subagents

### and the ability to plan
- The builtin **todo list** middleware

In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from util import pretty_print, print_todos, print_exchange, print_activity
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent

model = "claude-haiku-4-5-20251001"
research_system_prompt = """You are an expert research assistant."""

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
)

result = agent.invoke({"messages": [{"role": "user", "content": "What tools do you have?"}]})
pretty_print(result)

### Human

What tools do you have?

### Ai

I have access to these tools:

1. **File Operations**
   - `ls` — List files in a directory
   - `read_file` — Read file contents (with pagination for large files)
   - `write_file` — Create new files
   - `edit_file` — Edit existing files
   - `glob` — Find files matching a pattern
   - `grep` — Search for text within files

2. **Task Management**
   - `write_todos` — Create and manage a structured task list for organizing complex work

3. **Delegation**
   - `task` — Launch subagents to handle isolated, complex tasks autonomously (general-purpose agent available)

These tools let me navigate filesystems, read and modify code/content, organize work, and delegate independent tasks to specialized agents when needed.

# The TODO list in action

#### Ask a deep agent to perform a complex task and it starts by planning out all the steps.

In [20]:
result = agent.invoke({"messages": [{"role": "user", "content": "I need to bake a cake. Make a plan"}]})
print_todos(result)

### Todo list

- ☐ Gather ingredients (flour, sugar, eggs, butter, baking powder, vanilla, salt, milk) — *pending*
- ☐ Gather equipment (mixing bowl, measuring cups/spoons, whisk, cake pan, oven) — *pending*
- ☐ Preheat oven to recipe temperature (typically 350°F) — *pending*
- ☐ Prepare cake pan (grease and flour, or use parchment paper) — *pending*
- ☐ Mix dry ingredients (flour, baking powder, salt) — *pending*
- ☐ Cream butter and sugar together until light and fluffy — *pending*
- ☐ Beat in eggs one at a time — *pending*
- ☐ Add vanilla extract — *pending*
- ☐ Alternate adding dry ingredients and milk to wet mixture — *pending*
- ☐ Pour batter into prepared pan — *pending*
- ☐ Bake for 25-35 minutes until toothpick comes out clean — *pending*
- ☐ Cool cake in pan for 10 minutes — *pending*
- ☐ Turn cake out onto wire rack to cool completely — *pending*
- ☐ Frost or decorate as desired (optional) — *pending*

# Tools

#### Deep agents, as well as their subagents, can be extended with tools. Let's make ours more useful by allowing it to search the internet.

In [22]:
@tool
def search(query: str) -> str:
    """Search the internet for information

    Args:
        query (str): search query
    """
    results = TavilyClient().search(query)
    return results.get("results", [])

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    tools=[search]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What are the top 2 stories in the news today?"}]})
pretty_print(result)

### Human

What are the top 2 stories in the news today?

### Ai

I'll search for today's top news stories for you.

🔧 **search**(`{'query': 'top news today'}`)

### Tool

[{"url": "https://www.youtube.com/watch?v=-J96Kwpd2sE", "title": "This Morning's Top Headlines – July 29 | Morning News NOW", "content": "DOCTOR FAUCI ON AS WELL. >> ALMOST CERTAINLY WILL BE SOME FIREWORKS THERE TODAY ON CAPITOL HILL. MEGAN, THANK YOU SO MUCH. WORLD LEADERS AND WHITE HOUSE OFFICIALS WERE AT THE NATIONAL CATHEDRAL YESTERDAY FOR THE FUNERAL OF SOUTH CAROLINA REPUBLICAN SENATOR LINDSEY GRAHAM. PRESIDENT TRUMP, VICE PRESIDENT JD VANCE AND FOX PERSONALITY SEAN HANNITY WERE AMONG THOSE WHO EULOGIZED GRAHAM, ALL OF THEM PAYING RESPECTS TO GRAHAM'S WORK ETHIC AND REFLECTING ON HIS LEGACY IN THE SENATE. THE 71 YEAR OLD [...] POPE LEO'S CONCERT FOR PEACE. GOOD MORNING. IT'S GOOD TO BE WITH YOU. I'M JESSICA LAYTON IN FOR JOE AND SAVANNAH TODAY, AND WE START THIS HOUR WITH THE POWERFUL STORMS THAT ARE MAKING THEIR WAY ACROSS THE EASTERN PART OF THE U.S. IN NEW JERSEY, STRONG WINDS AND DRIVING RAIN TOOK DOWN TREES AND FLOODED STREETS. IN FACT, PEOPLE HAD TO PUSH THEIR CARS THROUGH THAT DEEP WATER. IN PATERSON, JUST EAST OF NEW YORK CITY, TOW TRUCKS LINED UP TO PULL STALLED CARS OUT OF THE STREET. THE MAYOR THERE SAYS [...] FORCES JOINTLY CONDUCTED STRIKES IN IRAQ AGAINST IRAN ALIGNED TERRORISTS, THOUGH JUST HOURS EARLIER, PRESIDENT TRUMP HAD TOUTED PROGRESS IN PEACE TALKS WHILE AGAIN THREATENING STRIKES AGAINST IRAN'S MAJOR BRIDGES AND POWER PLANTS IF THERE WAS NO LONG TERM AGREEMENT. TELLING FOX NEWS. >> I THINK WE HAVE A VERY STRONG POSITION RIGHT NOW. THEY KNOW I'M GOING TO DO THAT. IF THEY DON'T MAKE A DEAL, THE BRIDGES ARE GOING TO BE GONE. >> THE PRESIDENT ALSO MEETING FACE TO FACE AT THE WHITE HOUSE WITH", "score": 0.6551821, "raw_content": null}, {"url": "https://www.cbsnews.com", "title": "CBS News | Breaking news, top stories & today's latest ...", "content": "to aquariums across the U.S. in a multinational emergency operation after their home aquarium in Canada shuttered. Jared Ochacher reports.  Jul 25  2:57Image 84:  #### SpaceX launches first rocket since IPO Elon Musk's aerospace company, SpaceX, launched its massive Starship rocket in its first test flight since the company went public. Since achieving the largest IPO in history, SpaceX's stock has dropped drastically. Mark Strassmann takes a look.  Jul 25  3:19Image 85:  #### Wildfire smoke [...] #### Wildfire smoke continues to shroud U.S. Millions of people are enduring dangerous air quality as Canadian wildfire smoke shrouds the skies. Jonathan Overpeck, dean of the School for Environment and Sustainability at the University of Michigan, explains how climate change is worsening wildfires.  Jul 17  3:32Image 86:  #### How AI data centers impact the environment New York has become the first state to impose a temporary moratorium on data center development while state lawmakers lay the [...] that hit home contains \"alien world chemistry,\" experts say When scientists examined the preserved fragments of a meteorite that crashed in 2024, they found brine-like fluids and key molecules.  Jul 16Image 95: 55385926077-ef10cd6d45-k.jpg  #### Russian Soyuz rocket launches fresh crew to space station The new crew will replace three other Soyuz fliers wrapping up a 240-day stay aboard the International Space Station.  Jul 14Image 96: Interstellar Sugar  #### Astronomers discover sugar in space", "score": 0.59033114, "raw_content": null}, {"url": "https://apnews.com", "title": "Associated Press News: Breaking News, Latest Headlines and Videos ...", "content": "+ Indigenous peoples and climate\n  + Climate Questions\n  + Climate Migration\n  + India Focus\n   \n\n  SECTIONS\n\n  Indigenous peoples and climate Climate Questions Climate Migration India Focus\n\n  TOP STORIES\n\n  - Major Oregon interstate closed for 70 miles due to wildfire\n  - Life atop a mountain of garbage begins to change as Indonesia cuts methane emissions\n  - As Trump boosts nuclear power, regulators seek to eliminate a longstanding radiation safety practice\n Health\n\n  TOP STORIES [...] TOP STORIES\n\n  - Wall Street's flip from AI to less-loved stocks accelerates, while oil prices keep easing\n  - eBay agreed to pay nearly $50 million to couple sent cockroaches, bloody pig mask\n  - Workplaces look for cheaper AI as ‘tokenmaxxing’ fades as a corporate fad\n Science\n\n  + Space\n  + Animals\n  + The Ancient World\n  + Climate\n  + Medicine\n   \n\n  SECTIONS\n\n  Space Animals The Ancient World Climate Medicine\n\n  TOP STORIES [...] SECTIONS\n\n  Indigenous peoples and climate Climate Questions Climate Migration India Focus\n\n  TOP STORIES\n\n  + Major Oregon interstate closed for 70 miles due to wildfire\n  + Life atop a mountain of garbage begins to change as Indonesia cuts methane emissions\n  + As Trump boosts nuclear power, regulators seek to eliminate a longstanding radiation safety practice\n Health \n\n  TOP STORIES", "score": 0.5370662, "raw_content": null}, {"url": "https://abcnews.com", "title": "ABC News - Breaking News, Latest News and Videos", "content": "U.S.\n\n## France and Spain battle wildfires as thousands evacuated from homes\n\nU.S.\n\n## How lighthearted social media trend of 'teen takeovers' took a serious turn\n\nU.S.\n\n## 8 people, including 6 children, found dead after house fire, some with gunshot wounds\n\nTop Stories\n\n ### Two men parachute from Seattle's Space Needle\n\n ### Punch the monkey flooded with birthday messages\n\n ### Mangione prosecutors to lobby for stiff sentence\n\n ### Oregon fire looms near town; 4th firefighter dies [...] ## Iran live updates: Trump met with advisers on Iran options, sources say\n\nLive Updates\n\nSee All10+\n\n 9:41 PM\n\n  Army confirms identity of service member killed in Iranian attack in Jordan\n 6:17 PM\n\n  Trump met with advisers on Iran options today: Sources\n\nLive\n\nLive\n\n## 20/20: The Hunt for Mr. Right\n\n What Would You Do?3:00 AM\n 20/20: Where Are You Jacob?4:00 AM\n On The Red Carpet Presents: \"Moana\"6:00 AM\n GMA Life6:30 AM\n\nInternational [...] ### Trump-backed SAVE Act is newest effort to collect private voter information\n ### GOP uses New Jersey voting glitch to push to pass SAVE Act before Congress' recess\n ### A liberal lumberjack may soon be Maine's Democrat Senate nominee. Who's Troy Jackson?\n ### Obama surprises WNBA All-Stars at their practice at his Presidential Center\n\nPresident Trump\n\n## Trump's hunt for leverage over Saudi Arabia fuels nuclear proliferation concerns", "score": 0.46395484, "raw_content": null}, {"url": "https://www.cnn.com", "title": "CNN: Breaking News, Latest News and Videos", "content": "WASHINGTON, DC - JANUARY 29:  FIFA President Gianni Infantino holds the FIFA World Cup Trophy as he addresses the winter meeting of The U.S. Conference of Mayors on January 29, 2026 in Washington, DC. Infantino promoted the 2026 FIFA World Cup, which will be co-hosted by the United States, Canada, and Mexico, highlighting its potential benefits for host cities and communities. (Photo by Alex Wong/Getty Images) [...] Bryan Kohberger, who is accused of killing four University of Idaho students in November 2022, listens during his arraignment hearing in Latah County District Courtin Moscow, Idaho, on May 22, 2023.\nDr. Anthony Fauci, former Director of the National Institute of Allergy and Infectious Diseases, arrives to testify before the House Oversight and Accountability Committee Select Subcommittee on the Coronavirus Pandemic at the Rayburn House Office Building on June 03, 2024 in Washington, DC.", "score": 0.4504325, "raw_content": null}]

### Ai

Based on the search results, here are the top 2 stories in the news today:

1. **Severe Storms Hitting Eastern U.S.** — Powerful storms are moving across the eastern United States, bringing strong winds and heavy rain. In New Jersey, the weather has caused fallen trees and street flooding, with people having to push their cars through deep water. In Paterson, east of New York City, tow trucks are pulling stalled vehicles from flooded streets.

2. **Senator Lindsey Graham's Funeral** — World leaders and White House officials attended Senator Lindsey Graham's funeral at the National Cathedral. President Trump, Vice President JD Vance, and Fox personality Sean Hannity were among those who eulogized the 71-year-old South Carolina Republican senator, reflecting on his work ethic and legacy in the Senate.

# Task Delegation: Where Deep Agents Shine

#### The typical pattern is one main supervisor agent delegating to one or more specialized subagents.

### Each subagent has:
- its own context window
- its own tools
- its own system prompt
- its own model (can override the main agent's model)
- its own skills

### The main agent
- delegates via its task() tool
- sees only the final results of its subagents, keeping its context window clean

![Deep agent task delegation](images/deep-agent-delegation.svg)

# Defining Subagents: Dictionary Approach

#### Just define your subagent as a Python **dictionary** and plug it into your supervisor main agent.

In [25]:
research_subagent = {
    "name": "research-agent",
    "description": "Research a given task",
    "system_prompt": f"""You are a research assistant.
                         Use tools to gather information.
                         Structure findings with clear headings and inline citations.
                         Limit to 3 search calls.""",
    "model": "anthropic:claude-haiku-4-5-20251001",
    "tools": [search]
}

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[research_subagent])

# Defining Subagents: Compiled Approach

For more complex workflows we can use **compiled** subagents, allowing us arbitrary complexity.

In [24]:
from langchain.agents import create_agent
from deepagents import CompiledSubAgent

research_subagent = create_agent(
    model=model,
    tools=[search],
    system_prompt="""You are a research assistant.
                     Use the search tool to gather information.
                     Structure findings with clear headings and inline citations.
                     Limit to 3 search calls.""",
)

compiled_subagent = CompiledSubAgent(
    name="research-agent", description="Research a given task using web search", runnable=research_subagent,
)

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[compiled_subagent])

# Filesystems and backends

#### Deep agents are designed to handle long-running, complex tasks. They typically do this by decomposing tasks into sub-tasks and delegating execution to subagents. **Backends** expose a **filesystem** surface to allow (sub)agents to read and write state through a collection of tools: `ls, read_file, write_file, edit_file, glob, and grep`.

A few prebuilt filesystem backends that you can quickly use with your deep agent:

<table style="font-size:18px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:12px; text-align:left;">Backend</th>
<th style="border:1px solid #999; padding:12px; text-align:left;">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Default<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash")</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped. The default filesystem backend for an agent is stored in langgraph state. Files persist across turns within a thread (via your checkpointer) and are not shared across threads.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local filesystem persistence<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=FilesystemBackend(root_dir="/Users/nh/Desktop/"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the deep agent access to your local machine's filesystem. You can specify the root directory that the agent has access to. Any provided <code style="font-size:15px; color:#2b8a3e;">root_dir</code> must be an absolute path. Typically, wrap in a <code style="font-size:15px; color:#2b8a3e;">CompositeBackend</code> to keep internal agent data (offloaded tool results, conversation history) separate from your project files.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Durable store (LangGraph store)<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=StoreBackend())</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the agent access to long-term storage that is persisted across threads. Great for storing longer term memories or instructions that are applicable to the agent over multiple executions.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Context Hub<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=ContextHubBackend("my-agent"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Stores files durably in a LangSmith Hub repo, without provisioning a separate LangGraph store.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Sandbox<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=sandbox)</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Execute code in isolated environments. Sandboxes provide filesystem tools plus the <code style="font-size:15px; color:#2b8a3e;">execute</code> tool for running shell commands. Choose from LangSmith, AgentCore, Daytona, Deno, E2B, Modal, Runloop, or local VFS.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local shell<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=LocalShellBackend(root_dir=".", env={"PATH": "/usr/bin:/bin"}))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Filesystem and shell execution directly on the host. No isolation—use only in controlled development environments. See security considerations below.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Composite</td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped by default, <code style="font-size:15px; color:#2b8a3e;">/memories/</code> persisted across threads. The Composite backend is maximally flexible. You can specify different routes in the filesystem to point towards different backends.</td>
</tr>
</tbody>
</table>

# Default Backend
- Virtual and ephemeral filesystem
- Default backend is thread-scoped
- Without a `thread_id` the agent cannot find the referenced filesystem

####  Thread ID is how we tell our agent that our response is part of a larger conversation.

In [26]:
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, checkpointer=MemorySaver(),  # We don't have to mention the backend when using the 'default'; it is just available to us
)

config = {"configurable": {"thread_id": "joke-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Tell me a joke about computer scientists and write it to /joke.md"}]
}, config=config)
pretty_print(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

I'll write a joke about computer scientists to that file for you.

🔧 **write_file**(`{'file_path': '/joke.md', 'content': '# A Joke About Computer Scientists\n\nWhy do computer scientists make terrible lovers?\n\nBecause they keep trying to debug their partner\'s emotions and push their commits without asking for permission first.\n\n---\n\n*Bonus joke: A computer scientist is someone who looks at an empty glass and says, "At least it\'s consistent."*\n'}`)

### Tool

Updated file /joke.md

### Ai

Done! I've written a joke about computer scientists to `/joke.md`. It includes both a main joke and a bonus one about their perspective on things.

#### We are able to read the filesystem because we pass in the same `config` object has the `thread_id` we previously used.

In [28]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=config)
print_exchange(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

**Why do computer scientists make terrible lovers?**

Because they keep trying to debug their partner's emotions and push their commits without asking for permission first.

**Bonus:** A computer scientist is someone who looks at an empty glass and says, "At least it's consistent."

#### This time the agent cannot find the file because we use a different `thread_id`

In [30]:
other_config = {"configurable": {"thread_id": "other-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=other_config)
print_exchange(result)

### Human

Read /joke.md and tell me the joke

### Ai

The root directory appears to be empty. The `/joke.md` file doesn't exist. Would you like to create it, or did you mean a different path?

# Store Backend
- Long-term
- Backed by a database
- Persisted across threads
- Good for storing longer term memories or instructions that are applicable to the agent over multiple executions.

#### In this example we write a memory in a thread named 'session-a'

In [32]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=store,
)

# Write a memory in one thread...
session_a_config = {"configurable": {"thread_id": "session-a"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Remember that my name is Andrew by writing it to /profile.md"}]
}, config=session_a_config)

pretty_print(result)

### Human

Remember that my name is Andrew by writing it to /profile.md

### Ai

🔧 **write_file**(`{'file_path': '/profile.md', 'content': '# Profile\n\n**Name:** Andrew\n'}`)

### Tool

Updated file /profile.md

### Ai

Done. I've saved your name to /profile.md.

#### Now we can read the memory back from a _different_ thread. Unlike the default backend, the **StoreBackend** persists files across threads.

In [35]:
session_b_config = {"configurable": {"thread_id": "session-b"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /profile.md and tell me my name"}]
}, config=session_b_config)

print_exchange(result)

### Human

Read /profile.md and tell me my name

### Ai

Your name is **Andrew**.

# Store Backend: Choosing the backing database

The <code style="font-size:15px; color:#2b8a3e;">StoreBackend</code> is <b>storage-agnostic</b>. It is a thin filesystem adapter over whatever LangGraph <code style="font-size:15px; color:#2b8a3e;">BaseStore</code> you pass to <code style="font-size:15px; color:#2b8a3e;">create_deep_agent(store=...)</code>. The database is chosen entirely by that one argument &mdash; the agent code never changes.

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code>agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=<span style="color:#cf222e;">&lt;ANY BaseStore&gt;</span>,   <span style="color:#6e7781;"># &larr; this is the database choice</span>
)</code></pre>

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Store</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Package</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Import</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">InMemoryStore</code> <span style="color:#6e7781;">(dev only)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left;">built-in</td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.memory</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">PostgresStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-postgres</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.postgres</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">RedisStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-redis</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.redis</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">MongoDBStore</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph-store-mongodb</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph.store.mongodb</code></b></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;">Anything custom</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">your code</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">subclass <code style="color:#2b8a3e;">BaseStore</code></td>
</tr>
</tbody>
</table>

### MongoDB example

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code><span style="color:#cf222e;">from</span> langgraph.store.mongodb <span style="color:#cf222e;">import</span> MongoDBStore

<span style="color:#cf222e;">with</span> MongoDBStore.from_conn_string(
    conn_string=<span style="color:#0a3069;">"mongodb://localhost:27017"</span>,
    db_name=<span style="color:#0a3069;">"deepagents"</span>,
    collection_name=<span style="color:#0a3069;">"memories"</span>,
) <span style="color:#cf222e;">as</span> store:
    agent = create_deep_agent(
        model=model,
        system_prompt=research_system_prompt,
        backend=StoreBackend(namespace=<span style="color:#cf222e;">lambda</span> ctx: (<span style="color:#0a3069;">"memories"</span>,)),
        store=store,   <span style="color:#6e7781;"># only this line changed &mdash; files now persist in MongoDB</span>
    )
    <span style="color:#6e7781;"># ...same invoke() calls as the InMemoryStore cell</span></code></pre>


# Composite Backend
- Flexible: specify different routes in the filesystem to point towards different backends.

#### Example:
- Use <code style="font-size:15px; color:#2b8a3e;">StateBackend</code> for thread-scoped default storage
- Save persistent memories to <code style="font-size:15px; color:#2b8a3e;">/memories/</code>. These files are available from different threads.
- Save ephemeral memories to the root <code style="font-size:15px; color:#2b8a3e;">/</code>. These are not available across different threads.



In [36]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Route by path: everything is ephemeral (StateBackend) except /memories/, which is routed to the durable StoreBackend.
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("memories",))},
)

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, backend=composite_backend, store=store,
)

# Thread A: write one ephemeral file and one durable file
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md"
    )}]
}, config={"configurable": {"thread_id": "thread-a"}})
pretty_print(result)

### Human

Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md

### Ai

I'll create both files for you.

🔧 **write_file**(`{'file_path': '/draft.md', 'content': '# Scratch Notes\n\nQuick notes and ideas go here.\n'}`)

🔧 **write_file**(`{'file_path': '/memories/profile.md', 'content': '# Profile\n\n**Name:** Andrew\n'}`)

### Tool

Updated file /draft.md

### Tool

Updated file /memories/profile.md

### Ai

Done. Created `/draft.md` for scratch notes and saved your name to `/memories/profile.md`.

In [37]:
# Thread B: a *different* thread. The /memories/ route persisted; /draft.md did not.
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each."
    )}]
}, config={"configurable": {"thread_id": "thread-b"}})
print_exchange(result)

### Human

Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each.

### Ai

**Findings:**

- **/memories/profile.md**: Contains 3 lines. Shows a markdown header "# Profile" and a name field listing "Andrew".

- **/draft.md**: File not found. This file does not exist in the filesystem.

# Context Isolation vs. Shared Filesystems

When a supervisor delegates to a subagent, two different things happen to two different kinds of state:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left; width:22%;">What</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Behavior</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Message context</b><br><span style="color:#6e7781;">(isolated)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The subagent runs in its <b>own context window</b>. Its intermediate reasoning and tool calls never enter the supervisor's history &mdash; only a single, compact <b>final report</b> is returned. This is the whole point of subagents: keep the supervisor's context clean.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Filesystem</b><br><span style="color:#6e7781;">(shared)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The backend is <b>shared</b> between the supervisor and all subagents. Any file a subagent writes <b>remains in state after the subagent finishes</b> and stays readable by the supervisor and sibling subagents.</td>
</tr>
</tbody>
</table>

### The pattern this unlocks

The filesystem is a **shared side-channel** for moving large results across the subagent boundary *without* paying for them in context:

1. Supervisor delegates: *"research X and write the findings to <code style="color:#2b8a3e;">/research.md</code>"*
2. Subagent does the heavy work, writes the file, and returns just *"wrote results to /research.md"*
3. Supervisor (or another subagent) reads <code style="color:#2b8a3e;">/research.md</code> when it needs the detail

The bulky output travels through the filesystem; only a one-line handoff travels through the conversation.

<b>Caveat:</b> dictionary subagents inherit the supervisor's backend automatically. A fully custom <code style="font-size:15px; color:#2b8a3e;">CompiledSubAgent</code> graph has its own state and only shares the filesystem if you wire it to the same backend.

In [38]:
from langgraph.checkpoint.memory import MemorySaver

# A subagent whose job is to write its findings to a file (not to report them inline).
writer_subagent = {
    "name": "writer-agent",
    "description": "Writes the requested content to a file, then reports only the path.",
    "system_prompt": (
        "Write the requested content to the requested file path using write_file. "
        "Then report ONLY the path you wrote to — do not repeat the content."
    ),
    "tools": [],
}

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a supervisor. Delegate writing tasks to the writer-agent subagent via the task tool. "
        "The subagent shares your filesystem, so after it finishes you can read the file it wrote yourself."
    ),
    subagents=[writer_subagent],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "subagent-fs-demo"}}

# print_activity streams the run with subgraphs=True, so we can watch the *subagent*
# call write_file inside its own isolated context — that tool call never appears in
# the supervisor's returned messages.
print_activity(agent, (
    "Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. "
    "After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me."
), config=config)

### Activity timeline

- 🧑‍✈️ **supervisor** → 📨 delegates via **task** to `writer-agent`
    - 🤖 **writer-agent** → 🔧 **write_file**(`/poem.md`, content=“Waves crash on the shore Salty breezes fill the air Ocean's rhythm flows”)
    - 🤖 **writer-agent** ← 📥 `write_file` → Updated file /poem.md
- 🧑‍✈️ **supervisor** ← 📥 `task` → /poem.md
- 🧑‍✈️ **supervisor** → 🔧 **read_file**(`/poem.md`)
- 🧑‍✈️ **supervisor** ← 📥 `read_file` → 1 Waves crash on the shore 2 Salty breezes fill the air 3 Ocean's rhythm flows

#### Because we passed a checkpointer + thread_id, the run's state is durable on this thread. The subagent's file lives in the shared filesystem, readable by the main supervisor agent.

In [39]:
state = agent.get_state(config)
print("Files in shared state:", list(state.values.get("files", {}).keys()))

# And the supervisor's own transcript only ever saw the delegation + the final read —
# never the subagent's write_file call (that stayed isolated in the subagent's context).
pretty_print(state.values)

Files in shared state: ['/poem.md']


### Human

Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me.

### Ai

🔧 **task**(`{'subagent_type': 'writer-agent', 'description': 'Write a haiku about the ocean to the file /poem.md. A haiku has three lines with 5-7-5 syllables respectively. Return the path to the file you created.'}`)

### Tool

/poem.md

### Ai

Now let me read the file:

🔧 **read_file**(`{'file_path': '/poem.md'}`)

### Tool

     1	Waves crash on the shore
     2	Salty breezes fill the air
     3	Ocean's rhythm flows

### Ai

Here's the exact contents of /poem.md:

```
Waves crash on the shore
Salty breezes fill the air
Ocean's rhythm flows
```

# Context Management Techniques

### A. Subagent Context Isolation

Each subagent gets its own, isolated context window. This ensures that context relevant to a specific task is only seen by the subagent working on that task.

### B. Offloading Large Inputs and Outputs

When a tool returns (or receives) more than comfortably fits in context, the harness writes the payload to the filesystem and replaces it in the message history with a short **reference plus a preview** (e.g. the first 10 lines). The agent calls <code style="color:#2b8a3e;">read_file</code> to pull back more **only if it actually needs the detail** — instead of dragging thousands of lines through every subsequent turn.

<img src="images/offloading-results.png" alt="Offloading large tool results to the filesystem" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

<ul style="font-size:16px;">
<li>Oversized tool results are evicted and swapped for a reference.</li>
<li>The agent reads back just the slice it needs via <code style="color:#2b8a3e;">read_file</code>.</li>
</ul>

### C. Context Summarization

When the **conversation itself** approaches the model's limit (~85% of <code style="color:#2b8a3e;">max_input_tokens</code>), <code style="color:#2b8a3e;">SummarizationMiddleware</code> runs a **compaction step**: older turns are compressed into an LLM-generated summary — session intent, artifacts created, next steps — while the most recent ~10% of tokens are kept verbatim.

<ul style="font-size:16px;">
<li><b>In-context summary + pointer:</b> the summary replaces the old messages and tells the agent where the full record lives.</li>
<li><b>Filesystem preservation:</b> the original messages are written to <code style="color:#2b8a3e;">/conversation_history/{thread_id}.md</code>, so the agent can search or re-read them if needed.</li>
</ul>

<img src="images/summarization.png" alt="Summarizing older conversation turns into a compact summary + pointer" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

